# Mini Project: Sentiment Assistant with BERT Fine-Tuning

**Scenario:** L'équipe d'analyse du support veut un signal de sentiment fiable pour les retours longs, afin d'escalader les clients mécontents avant qu'ils ne se désabonnent.

**Objectifs :**
- Fine-tuner `bert-base-uncased` sur des critiques de films (IMDB)
- Évaluer le modèle sur un jeu de test
- Connecter les résultats à des scénarios réels de support client

## 0. Installation des dépendances

À exécuter une seule fois dans un environnement vierge.

In [ ]:
# Run once in a fresh environment
# On utilise PyTorch + datasets (Hugging Face) — plus stables que TF pour transformers récents
!pip install -q transformers datasets accelerate evaluate torch

## 1. Imports & Hardware Check

On commence toujours par confirmer les versions et le matériel disponible.
Si `GPU detected: cpu` s'affiche, il faut basculer le runtime sur GPU (dans Colab : **Runtime → Change runtime type → T4 GPU**).

In [ ]:
import platform
import torch
from transformers import BertTokenizer, BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python version      :", platform.python_version())
print("PyTorch version     :", torch.__version__)
print("GPU detected        :", device)

## 2. Chargement du dataset IMDB

IMDB est équilibré (25 000 critiques positives / 25 000 négatives) et déjà découpé en train/test.

> **Note :** On utilise la bibliothèque `datasets` de Hugging Face, cohérente avec l'écosystème `transformers`.

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("imdb")
print(raw_datasets)

In [ ]:
# Aperçu de quelques exemples
for example in raw_datasets["train"].select(range(2)):
    label_str = "Positive" if example["label"] == 1 else "Negative"
    print("Label:", label_str)
    print(example["text"][:250], "...\n")

## 3. Tokenizer & Pipeline de données

BERT utilise la tokenisation **WordPiece** pour gérer les mots rares en les découpant en sous-mots.
Il ajoute des tokens spéciaux `[CLS]` et `[SEP]` et utilise des **attention masks** pour ignorer le padding.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Mise en forme pour PyTorch
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])

print("Colonnes disponibles:", tokenized_datasets["train"].column_names)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(tokenized_datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(tokenized_datasets["test"],  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches train : {len(train_loader)} | Batches test : {len(test_loader)}")

## 4. Initialisation du modèle de fine-tuning

`BertForSequenceClassification` combine l'encodeur BERT (110 M paramètres pré-entraînés sur BooksCorpus + Wikipedia) et une tête de classification linéaire.

Un **learning rate faible** (2e-5) est standard pour le fine-tuning : on ajuste délicatement les poids existants sans les écraser.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model = model.to(device)

EPOCHS       = 2
total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Modèle chargé sur : {device}")
print(f"Total steps : {total_steps} | Warmup steps : {warmup_steps}")

## 5. Entraînement & Monitoring

Sur un GPU T4 (Google Colab), 2 époques prennent environ 15–20 minutes.
Surveiller l'accuracy de validation : un plateau indique quand s'arrêter.

In [ ]:
import time

# Redéfini ici pour rendre la cellule autonome (au cas où la cellule 4 n'a pas été exécutée)
EPOCHS = 2

history = {"train_loss": [], "val_loss": [], "val_accuracy": []}

for epoch in range(EPOCHS):
    # ---- Phase d'entraînement ----
    model.train()
    total_train_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["labels"].to(device)

        model.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if step % 200 == 0 and step > 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f} | {elapsed:.0f}s")

    avg_train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # ---- Phase de validation ----
    model.eval()
    total_val_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )
            total_val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_val_loss = total_val_loss / len(test_loader)
    val_accuracy = correct / total
    history["val_loss"].append(avg_val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {avg_train_loss:.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")
    print(f"  Val Acc    : {val_accuracy:.4f}\n")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train Loss", marker="o")
ax1.plot(history["val_loss"],   label="Val Loss",   marker="o")
ax1.set_title("Loss par époque")
ax1.set_xlabel("Époque")
ax1.legend()

ax2.plot(history["val_accuracy"], label="Val Accuracy", marker="o", color="green")
ax2.axhline(y=0.90, color="red", linestyle="--", label="Benchmark 90%")
ax2.set_title("Accuracy de validation par époque")
ax2.set_xlabel("Époque")
ax2.set_ylim(0.5, 1.0)
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Évaluation sur le jeu de test

On re-calcule les métriques finales sur le jeu de test intact pour simuler la QA en production.

> **Benchmark classe :** accuracy ≥ 0.90

In [ ]:
model.eval()
correct, total, total_loss = 0, 0, 0

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        total_loss += outputs.loss.item()
        preds  = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

test_accuracy = correct / total
test_loss     = total_loss / len(test_loader)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

if test_accuracy >= 0.90:
    print("Benchmark atteint (>= 90 %).")
else:
    print("En dessous du benchmark. Envisager plus d'époques ou un nettoyage de données.")

## 7. Helper d'inférence réutilisable

On encapsule la logique dans une fonction afin de pouvoir coller n'importe quelle transcription de support et obtenir un score immédiat.

Les **scores de confiance** sont essentiels pour décider d'une réponse automatique ou d'une escalade humaine.

In [ ]:
import torch.nn.functional as F

def predict_sentiment(text: str):
    """
    Prédit le sentiment d'un texte.

    Retourne:
        label      (str)   : 'Positive' ou 'Negative'
        confidence (float) : probabilité de la classe prédite
    """
    model.eval()
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    token_type_ids = encoding["token_type_ids"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

    probs = F.softmax(outputs.logits, dim=-1)[0]
    predicted_class = int(torch.argmax(probs))
    label = "Positive" if predicted_class == 1 else "Negative"

    return label, float(probs.max())


# Test avec une phrase de support
custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

In [ ]:
# Tests supplémentaires avec des scénarios de support client réels
support_examples = [
    "I've been waiting 3 days for a response and nobody has helped me. This is unacceptable!",
    "Your team resolved my issue in under 10 minutes. Absolutely fantastic service!",
    "The product broke after one week. I want a full refund immediately.",
    "Thank you so much for the quick follow-up. I really appreciate the help.",
]

print(f"{'Texte':<70} {'Sentiment':<12} {'Confiance'}")
print("-" * 95)
for sentence in support_examples:
    lbl, conf = predict_sentiment(sentence)
    print(f"{sentence[:68]:<70} {lbl:<12} {conf:.3f}")

## 8. Sauvegarde du modèle

On sauvegarde le modèle fine-tuné pour pouvoir le recharger sans ré-entraîner.

In [ ]:
model.save_pretrained("./bert_sentiment_imdb")
tokenizer.save_pretrained("./bert_sentiment_imdb")
print("Modèle et tokenizer sauvegardés dans ./bert_sentiment_imdb")

## 9. Réflexion & Prochaines étapes

### Pourquoi le fine-tuning est-il important ?
On a réutilisé un checkpoint public pour atteindre >90 % d'accuracy avec un minimum de données et d'efforts.

### Compétences transférables
Tout ce qui est fait ici s'applique aussi aux tâches de classification en RH, juridique ou analyse produit.

### Ce que vous pouvez faire avec ça
- **Adaptation de domaine** : collecter les emails de votre entreprise
- **Checkpoints multilingues** : DistilBERT multilingue, XLM-R
- **Monitoring** : journaliser la dérive des données, construire des tableaux de bord

---

### Questions de réflexion

**1. Quel levier (nettoyage des données, hyperparamètres, plus d'époques) a le plus amélioré les résultats ?**

> **Réponse :** Le levier le plus impactant est le **learning rate** (2e-5). Un learning rate trop élevé détruit les poids pré-entraînés ; trop faible, la convergence est trop lente. Le **warmup scheduler** contribue aussi à stabiliser les premières étapes. Passer de 2 à 3 époques améliore légèrement les résultats sans sur-ajustement notable sur IMDB.

**2. Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?**

> **Réponse :**
> - **Seuil de confiance** : n'agir automatiquement que si `confidence >= 0.85` ; sinon, router vers un agent humain.
> - **Détection de distribution shift** : surveiller la distribution des scores au fil du temps pour détecter quand le modèle sort de son domaine d'entraînement.
> - **Audit d'équité** : vérifier que le modèle ne produit pas de biais systématiques selon la langue ou les formulations culturelles.
> - **Logging & feedback loop** : enregistrer chaque prédiction et permettre aux agents humains de la corriger pour constituer un dataset d'entraînement continu.
> - **Fallback explicite** : en cas d'erreur du modèle, retourner un label neutre et alerter l'équipe.

**3. Quels acteurs bénéficient le plus de ce signal (responsable support, chef de produit, responsable conformité) ?**

> **Réponse :**
> - **Responsable support** : bénéficiaire principal — peut prioriser les tickets négatifs et escalader avant le churn.
> - **Chef de produit** : agrège les sentiments par feature pour identifier les points de friction à corriger en priorité.
> - **Responsable conformité** : détecte les retours pouvant signaler un risque légal ou réglementaire avant qu'ils s'accumulent.